# Session 1, Part 2: cost and environmental impacts (TEA and LCA)

Run each cell with **Shift+Enter**, from the top. Cells marked **Your turn** are exercises, at level **1** (change a value), **2** (a few lines of code), or **3** (advanced, optional). Every exercise cell already runs as is. Each exercise has a hint, and a solution that stays hidden until you click **Show solution**.

QSDsan joins three pieces: a simulated `System` (mass and energy balances), a `TEA` that turns the system's equipment, utilities, and flows into cost metrics, and an `LCA` that turns the system's inventory (construction, transport, streams, electricity) into environmental impacts. Both read the **simulated** system, so you change the design or the assumptions and then ask again.

If you know SuperPro Designer or similar tools: `TEA` is the economics sheet and `LCA` is the inventory-and-characterization step that you would otherwise do in an LCA tool.

In [ ]:
# Google Colab only: installs QSDsan the first time. On your own computer this cell does nothing.
# After the install, restart the session (Runtime, Restart session) and run the notebook again from the top.
import sys
if 'google.colab' in sys.modules:
    try:
        import qsdsan
    except ImportError:
        %pip install -q qsdsan==1.5.3 exposan==1.5.3
        print('Installed. Now restart the session (Runtime, Restart session) and run the notebook again from the top.')
    else:
        import biosteam as bst
        bst.preferences.graphviz_format = 'png'   # Colab does not draw the default SVG diagrams

In [ ]:
import qsdsan as qs
from qsdsan.utils import create_example_treatment_systems
print(f'QSDsan {qs.__version__}')

qs.main_flowsheet.set_flowsheet('s1p2')   # keeps this notebook's objects separate from other work
qs.PowerUtility.price = 0.08              # electricity price, USD/kWh

## 1. The case: two ways to treat municipal wastewater

We compare two plants that treat the same wastewater (4,000 m3/day, COD of about 430 mg/L), taken from the QSDsan tutorials:

- an **aerobic** activated sludge plant: dependable, but it uses electricity for aeration and makes a lot of sludge;
- an **anaerobic** plant: it recovers methane (biogas), but it must be heated and dosed with alkalinity (sodium bicarbonate).

In [ ]:
aer_sys, ana_sys = create_example_treatment_systems()
aer_sys.simulate()
ana_sys.simulate()
aer, ana = aer_sys.units[0], ana_sys.units[0]
aer_sys.diagram()

In [ ]:
aer.results()

## 2. Techno-economic analysis (TEA)

A `TEA` is linked to a system. Its main inputs are the discount rate and the lifetime. It sorts every cash flow into categories: **capital** (installed equipment), **utilities** (electricity, heating), **materials** (purchased chemicals), **unit operating costs** (`add_OPEX`, here sludge disposal), and **sales** (products, or a treatment fee received).

In [ ]:
tea_aer = qs.TEA(aer_sys, discount_rate=0.05, lifetime=20)
tea_ana = qs.TEA(ana_sys, discount_rate=0.05, lifetime=20)
tea_aer.show()

In [ ]:
def breakdown(name, tea):
    print(f'{name:>10}: CAPEX {tea.CAPEX:>11,.0f} | material {tea.material_cost:>9,.0f} | '
          f'utility {tea.utility_cost:>9,.0f} | add_OPEX {tea.unit_add_OPEX:>9,.0f} | '
          f'sales {tea.sales:>9,.0f} | annual operating cost {tea.AOC:>10,.0f}')

print('CAPEX in USD, everything else in USD/yr')
breakdown('aerobic', tea_aer)
breakdown('anaerobic', tea_ana)

### How a unit's cost becomes the capital cost

Each unit fills `baseline_purchase_costs` in its `_cost` method (the equipment as bought). A **bare-module factor** `F_BM` turns the purchase cost into an **installed cost** (adding installation, piping, instrumentation). The `TEA` sums the installed costs of all units. The two example plants use one lump sum for the whole plant, so the bare-module factor is 1:

In [ ]:
for u in (aer, ana):
    print(f'{u.ID}: purchase {u.purchase_cost:,.0f} USD, F_BM {u.F_BM}, installed {u.installed_cost:,.0f} USD')
print(f'TEA installed equipment cost (aerobic): {tea_aer.installed_equipment_cost:,.0f} USD = CAPEX {tea_aer.CAPEX:,.0f} USD')

For most equipment you want the cost to depend on **size**. The `cost` decorator does this with the usual scaling relation: a reference size `S`, the cost at that size, and an exponent `n` (the "six-tenths rule" has `n` around 0.6), adjusted from the cost index of the reference year to the current one (`qs.CEPCI`).

Below, a digester costs 500,000 USD at 1,000 m3 (in 2017 dollars), scales with an exponent of 0.6, and has a bare-module factor of 2.

In [ ]:
from qsdsan.utils import cost

@cost('Volume', 'Tank', CEPCI=qs.CEPCI_by_year[2017], S=1000, cost=500000, n=0.6, BM=2.0)
class Digester(qs.SanUnit):
    _N_ins = _N_outs = 1
    _units = {'Volume': 'm3'}
    def __init__(self, ID='', ins=None, outs=(), thermo=None, HRT=20):
        qs.SanUnit.__init__(self, ID, ins, outs, thermo)
        self.HRT = HRT   # days
    def _run(self):
        self.outs[0].copy_like(self.ins[0])
    def _design(self):
        self.design_results['Volume'] = self.ins[0].F_vol * 24 * self.HRT   # m3

feed = qs.WasteStream('dig_feed', H2O=1000, units='m3/d')   # a stand-in influent
dig = Digester('D1', ins=feed)
for q in (50, 100, 200):                       # m3/day
    feed.ivol['H2O'] = q / 24                  # m3/hr
    dig.simulate()
    print(f"flow {q:>3} m3/d -> volume {dig.design_results['Volume']:>6.0f} m3, "
          f"purchase {dig.purchase_cost:>9,.0f} USD, installed {dig.installed_cost:>9,.0f} USD")

Doubling the size does not double the cost (2 to the power 0.6 is about 1.5). The cost index also matters: the same equipment costs more in later years. QSDsan's default `qs.CEPCI` is the 2017 value, so costs from decorators are in 2017 dollars unless you set it.

In [ ]:
print(f'qs.CEPCI is {qs.CEPCI} (2017 value: {qs.CEPCI_by_year[2017]}, 2023 value: {qs.CEPCI_by_year[2023]})')
dig.simulate()
print(f'Installed cost with the 2017 index: {dig.installed_cost:,.0f} USD')

qs.CEPCI = qs.CEPCI_by_year[2023]   # set the year you want, explicitly
dig.simulate()
print(f'Installed cost with the 2023 index: {dig.installed_cost:,.0f} USD')

qs.CEPCI = qs.CEPCI_by_year[2017]   # back to the default for the rest of the notebook

> **By default `qs.TEA` treats installed equipment cost as the total capital investment.** A full plant estimate adds markups (engineering, contingency, working capital, land). To include them, write a `TEA` subclass (tutorial 7, section 2). Real units in `qsdsan/unit_operations/` and in EXPOsan systems use the same `_design` and `_cost` pattern as above.

### Financial metrics

All metrics come from a discounted cash flow. **NPV** is the net present value of all cash flows over the lifetime (more negative means more expensive for a plant that only has costs). **EAC** is the equivalent annual cost. `solve_price` finds the price of a stream that brings the NPV to zero. For a treatment plant that is the **break-even treatment fee**.

In [ ]:
def summary(name, tea, unit):
    fee = -tea.solve_price(unit.ins[0]) * 1000   # USD per m3 of wastewater (a negative price means the plant is paid)
    print(f'{name:>10}: NPV {tea.NPV:>13,.0f} USD | EAC {tea.EAC:>11,.0f} USD/yr | break-even fee {fee:5.2f} USD/m3')

summary('aerobic', tea_aer, aer)
summary('anaerobic', tea_ana, ana)

The aerobic plant is cheaper here: the anaerobic plant must heat the whole flow, and the biogas from dilute wastewater does not pay for that. (Anaerobic treatment wins for stronger wastewater, which you can test in tutorial 7.)

**Your turn 1 (level 1).** The discount rate is an assumption. Print the NPV of both plants at 3%, 5%, and 10%. Does the ranking of the plants change?

<details><summary>Hint</summary>

Set `discount_rate` on both `TEA` objects and read `NPV`. The plant with the higher (less negative) NPV is cheaper.

</details>

In [ ]:
for r in (0.03, 0.05, 0.10):
    tea_aer.discount_rate = r
    # Your turn: set the same rate for tea_ana, and print both NPVs
    print(f'discount rate {r:.0%}: aerobic NPV {tea_aer.NPV:,.0f} USD')
tea_aer.discount_rate = 0.05   # restore

<details><summary>Show solution</summary>

```python
for r in (0.03, 0.05, 0.10):
    tea_aer.discount_rate = r
    tea_ana.discount_rate = r
    print(f'discount rate {r:.0%}: aerobic NPV {tea_aer.NPV:,.0f} USD, anaerobic NPV {tea_ana.NPV:,.0f} USD')
tea_aer.discount_rate = 0.05   # restore
tea_ana.discount_rate = 0.05
```

</details>

**Your turn 2 (level 2).** The plant capital (5 million USD for each) is an assumption. Raise the anaerobic plant's capital cost to 8 and 12 million USD (`ana.plant_capital`), re-simulate that system, and find its break-even fee each time. How much does the capital cost change the fee, and does it change which plant is cheaper? Restore the value at the end.

<details><summary>Hint</summary>

Changing `ana.plant_capital` changes the cost only after `ana_sys.simulate()`, because `_cost` runs during simulation. Compare with the aerobic fee from the summary above.

</details>

In [ ]:
for capital in (5e6,):                  # Your turn: add 8e6 and 12e6
    ana.plant_capital = capital
    ana_sys.simulate()                  # re-run so the new cost is used
    fee = -tea_ana.solve_price(ana.ins[0]) * 1000
    print(f'anaerobic capital {capital:,.0f} USD -> break-even fee {fee:.2f} USD/m3')
ana.plant_capital = 5e6
ana_sys.simulate()

<details><summary>Show solution</summary>

```python
for capital in (5e6, 8e6, 12e6):
    ana.plant_capital = capital
    ana_sys.simulate()                  # re-run so the new cost is used
    fee = -tea_ana.solve_price(ana.ins[0]) * 1000
    print(f'anaerobic capital {capital:,.0f} USD -> break-even fee {fee:.2f} USD/m3')
ana.plant_capital = 5e6
ana_sys.simulate()
aer_fee = -tea_aer.solve_price(aer.ins[0]) * 1000
print(f'aerobic (5,000,000 USD) break-even fee: {aer_fee:.2f} USD/m3')
```

</details>

## 3. Life cycle assessment (LCA)

An `LCA` is also linked to a system. It combines four kinds of inventory with **impact items** that hold characterization factors (CFs: the impact per unit of an item):

- **construction**: materials in the equipment (steel, concrete), counted once per lifetime of the equipment;
- **transportation**: moving things (here, sludge to a landfill);
- **streams**: material inputs and emissions linked to streams (positive = emission, negative = avoided impact);
- **other**: anything else, such as electricity.

Impacts are reported for **indicators** you choose. We use global warming (GWP) and fossil energy consumption (FEC).

In [ ]:
GWP = qs.ImpactIndicator('GlobalWarming', alias='GWP', unit='kg CO2-eq')
FEC = qs.ImpactIndicator('FossilEnergyConsumption', alias='FEC', unit='MJ')

# Impact items with characterization factors per functional unit (illustrative values)
Concrete = qs.ImpactItem('Concrete', functional_unit='m3', GWP=300., FEC=900.)
Steel = qs.ImpactItem('Steel', functional_unit='kg', GWP=2., FEC=25.)
Trucking = qs.ImpactItem('Trucking', functional_unit='kg*km', GWP=1e-4, FEC=2e-3)
electricity = qs.ImpactItem('electricity', functional_unit='kWh', GWP=0.5, FEC=5.93)
Concrete.show()

In [ ]:
# Construction: the anaerobic plant has the larger concrete digester
for u, concrete_m3, steel_kg in ((aer, 300, 30000), (ana, 400, 40000)):
    u.construction = [
        qs.Construction(linked_unit=u, item=Concrete, quantity=concrete_m3, quantity_unit='m3', lifetime=30),
        qs.Construction(linked_unit=u, item=Steel, quantity=steel_kg, quantity_unit='kg', lifetime=30),
    ]

# Transportation: each plant trucks its sludge 50 km, one load of the day's sludge per day
for u in (aer, ana):
    u.transportation = [qs.Transportation(
        linked_unit=u, item=Trucking, load_type='mass', load=u.outs[1].F_mass * 24, load_unit='kg',
        distance=50, distance_unit='km', interval=1, interval_unit='day')]

# Streams: sludge handling emits; biogas displaces natural gas (a credit, so negative)
qs.StreamImpactItem(linked_stream=aer.outs[1], GWP=0.2)
qs.StreamImpactItem(linked_stream=ana.outs[1], GWP=0.2)
qs.StreamImpactItem(linked_stream=ana.outs[2], GWP=-2.0, FEC=-50.)

Electricity is an "other" item. Its quantity depends on the simulation (the aerobic plant aerates), so we pass it as a **function**, not a number. The `LCA` calls the function whenever you ask for results, so it follows the current state of the system. If you pass a plain number, it is frozen at the baseline and your later changes (and any uncertainty analysis) will not reach the electricity term.

In [ ]:
lifetime = 20
lca_aer = qs.LCA(system=aer_sys, lifetime=lifetime, simulate_system=False,
                 electricity=lambda: aer.power_utility.rate * 24 * 365 * lifetime)
lca_ana = qs.LCA(system=ana_sys, lifetime=lifetime, simulate_system=False,
                 electricity=lambda: ana.power_utility.rate * 24 * 365 * lifetime)
lca_aer.show()

In [ ]:
import pandas as pd
rows = []
for name, lca in (('aerobic', lca_aer), ('anaerobic', lca_ana)):
    rows.append({'plant': name,
                 'construction': lca.total_construction_impacts['GlobalWarming'],
                 'transport': lca.total_transportation_impacts['GlobalWarming'],
                 'streams': lca.total_stream_impacts['GlobalWarming'],
                 'electricity': lca.total_other_impacts['GlobalWarming'],
                 'total': lca.get_total_impacts()['GlobalWarming']})
pd.DataFrame(rows).set_index('plant').round(0).rename_axis(columns='GWP over 20 years (kg CO2-eq)')

In [ ]:
# Divide by the service provided, so that different plants can be compared (per m3 of wastewater treated)
for name, lca, u in (('aerobic', lca_aer, aer), ('anaerobic', lca_ana, ana)):
    gwp = lca.get_normalized_impacts(u.ins[0], normalize_by='volume')['GlobalWarming']
    print(f'{name:>10}: {gwp:6.3f} kg CO2-eq per m3 treated')

**Your turn 3 (level 1).** The carbon intensity of electricity (0.5 kg CO2-eq per kWh above) depends on the grid. Set it to 0.1 (a clean grid) and compare the total GWP of both plants with before. Which plant is affected more, and why? Restore the value at the end.

<details><summary>Hint</summary>

Characterization factors live in the item's `CFs` dictionary, for example `electricity.CFs['GlobalWarming'] = ...`. In this model only the aerobic plant uses electricity (the anaerobic plant is heated with steam, which is not in this LCA), so only its footprint depends on the grid.

</details>

In [ ]:
before = {n: l.get_total_impacts()['GlobalWarming'] for n, l in (('aerobic', lca_aer), ('anaerobic', lca_ana))}
# Your turn: change the GWP characterization factor of the electricity item.
after = {n: l.get_total_impacts()['GlobalWarming'] for n, l in (('aerobic', lca_aer), ('anaerobic', lca_ana))}
for n in before:
    print(f'{n:>10}: {before[n]:>12,.0f} -> {after[n]:>12,.0f} kg CO2-eq')

<details><summary>Show solution</summary>

```python
before = {n: l.get_total_impacts()['GlobalWarming'] for n, l in (('aerobic', lca_aer), ('anaerobic', lca_ana))}
electricity.CFs['GlobalWarming'] = 0.1
after = {n: l.get_total_impacts()['GlobalWarming'] for n, l in (('aerobic', lca_aer), ('anaerobic', lca_ana))}
for n in before:
    print(f'{n:>10}: {before[n]:>12,.0f} -> {after[n]:>12,.0f} kg CO2-eq')
electricity.CFs['GlobalWarming'] = 0.5   # restore
```

</details>

## 4. Cost and carbon together

Which plant is better depends on what you weigh. Both plants treat the same wastewater and both metrics are per m3 treated, so the rows can be compared directly.

In [ ]:
rows = []
for name, tea, lca, u in (('aerobic', tea_aer, lca_aer, aer), ('anaerobic', tea_ana, lca_ana, ana)):
    fee = -tea.solve_price(u.ins[0]) * 1000
    gwp = lca.get_normalized_impacts(u.ins[0], normalize_by='volume')['GlobalWarming']
    rows.append((name, round(fee, 2), round(gwp, 3)))
tradeoff = pd.DataFrame(rows, columns=['plant', 'break-even fee (USD/m3)', 'GWP (kg CO2-eq/m3)'])
tradeoff

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.scatter(tradeoff.iloc[:, 1], tradeoff.iloc[:, 2])
for _, r in tradeoff.iterrows():
    ax.annotate(r['plant'], (r.iloc[1], r.iloc[2]), textcoords='offset points', xytext=(6, 4))
ax.set_xlabel('Break-even fee (USD/m3)')
ax.set_ylabel('GWP (kg CO2-eq/m3)')
ax.axhline(0, color='0.8', lw=0.8)
fig.tight_layout()

The aerobic plant costs less; the anaerobic plant has the lower (here negative) carbon footprint thanks to the biogas credit. No option wins on both, so the choice depends on the weight you give each, on the context (wastewater strength, energy prices, grid), and on how uncertain the inputs are. Session 2 shows how to quantify that uncertainty.

**Your turn 4 (level 2).** The anaerobic plant doses sodium bicarbonate (`ana.ins[1]`), but we gave that stream no environmental impact. Add a `StreamImpactItem` for it with a GWP of 0.6 kg CO2-eq per kg. Then create a new `LCA` for `ana_sys` (see the hint) and recompute the anaerobic GWP per m3 treated. Does the ranking on carbon change?

<details><summary>Hint</summary>

An `LCA` collects the stream impact items when it is created, so an item added afterwards does not enter an existing `LCA`. Create a new one. The impact of a stream is its flow times the characterization factor. Use `qs.StreamImpactItem(linked_stream=..., GWP=...)`.

</details>

In [ ]:
bicarbonate = ana.ins[1]
# Your turn: link an impact item to this stream, then create a new LCA the same way as lca_ana above.
gwp = lca_ana.get_normalized_impacts(ana.ins[0], normalize_by='volume')['GlobalWarming']
print(f'anaerobic GWP: {gwp:.3f} kg CO2-eq per m3 treated')

<details><summary>Show solution</summary>

```python
bicarbonate = ana.ins[1]
qs.StreamImpactItem(linked_stream=bicarbonate, GWP=0.6)
lca_ana2 = qs.LCA(system=ana_sys, lifetime=lifetime, simulate_system=False,
                  electricity=lambda: ana.power_utility.rate * 24 * 365 * lifetime)
gwp = lca_ana2.get_normalized_impacts(ana.ins[0], normalize_by='volume')['GlobalWarming']
print(f'anaerobic GWP: {gwp:.3f} kg CO2-eq per m3 treated (still lower than aerobic: {lca_aer.get_normalized_impacts(aer.ins[0], normalize_by="volume")["GlobalWarming"]:.3f})')
```

</details>

## Pitfalls to remember

- **Set `qs.CEPCI` explicitly** at the top of every analysis. Costs scale with it, and it can drift between analyses.
- **Compare the right capital number.** `purchase_cost` is the equipment as bought, `installed_cost` is purchase cost times `F_BM`, and the TEA total can add markups (tutorial 7).
- **Pass a function for quantities that change with the simulation** (electricity, chemical doses), as we did for electricity, so that the LCA does not freeze the baseline value.

## Where to go next

- Tutorials: [TEA](https://qsdsan.readthedocs.io/en/latest/tutorials/7_TEA.html), [LCA](https://qsdsan.readthedocs.io/en/latest/tutorials/8_LCA.html), [Modeling notes and pitfalls](https://qsdsan.readthedocs.io/en/latest/tutorials/14_Modeling_Notes_and_Pitfalls.html).
- Next session: uncertainty and sensitivity analysis on this same pair of plants, and dynamic simulation.